In [73]:
import sys
from datetime import datetime, timedelta
from pathlib import Path

import chardet
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow
import seaborn as sns
from folium.plugins import HeatMap, MarkerCluster

# adds parent file of the current directory
# to the paths in which Python looks for modules to import
# in the current Python process
sys.path.append(str(Path.cwd().parent))

from src.config import DATA_RAW_DIR, GEO_DATA_RAW_DIR, GEO_DATA_CLEAN_DIR
from utils.cleaning_utils import delete, normalize_columns_names, normalize_text_columns_cells, optimize_numeric_column, fill_rate
from utils.analysis_utils import plot_missing_bar, plot_numeric_histograms, plot_corr_heatmap, plot_qualitative


In [74]:
pd.set_option('display.max_columns', None)

In [75]:
# List all CSV files in DATA_RAW
ALL_DATA_FILES = list(iter(DATA_RAW_DIR.glob('*.csv')))

# List all recent CSV files in DATA_RAW, post 2006, without 2003 file
POST_2005_DATA_FILES = list(iter(DATA_RAW_DIR.glob('Incendies20*.csv')))
POST_2005_DATA_FILES = [f for f in POST_2005_DATA_FILES if "2003" and "2001" not in str(f)]

RAW_DATA_FILE = GEO_DATA_RAW_DIR / 'communes-france-2025.csv'

RAW_DATA_FILE

PosixPath('/home/coule/Documents/projets/incendies/data/data_geo_raw/communes-france-2025.csv')

In [76]:
# Check the encoding of the CSV files
with open(RAW_DATA_FILE, 'rb') as file:
    encodage = chardet.detect(file.read(10000))

print(encodage)

{'encoding': 'utf-8', 'confidence': 0.833516, 'language': 'fr', 'mime_type': 'text/plain'}


In [77]:
df = pd.read_csv(RAW_DATA_FILE, encoding=encodage["encoding"], dtype_backend='numpy_nullable')

/tmp/ipykernel_27541/650886494.py:1: DtypeWarning: Columns (0: code_insee, 1: dep_code, 2: canton_code, 3: epci_code, 4: code_insee_centre_zone_emploi, 5: code_unite_urbaine) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_DATA_FILE, encoding=encodage["encoding"], dtype_backend='numpy_nullable')


In [78]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34935 entries, 0 to 34934
Data columns (total 47 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   numero                             34935 non-null  Int64  
 1   code_insee                         34935 non-null  object 
 2   nom_standard                       34935 non-null  string 
 3   nom_sans_pronom                    34935 non-null  string 
 4   nom_a                              34935 non-null  string 
 5   nom_de                             34935 non-null  string 
 6   nom_sans_accent                    34935 non-null  string 
 7   nom_standard_majuscule             34935 non-null  string 
 8   typecom                            34935 non-null  string 
 9   typecom_texte                      34935 non-null  string 
 10  reg_code                           34935 non-null  Int64  
 11  reg_nom                            34935 non-null  string 
 12  d

In [79]:
df.columns

Index(['numero', 'code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a',
       'nom_de', 'nom_sans_accent', 'nom_standard_majuscule', 'typecom',
       'typecom_texte', 'reg_code', 'reg_nom', 'dep_code', 'dep_nom',
       'canton_code', 'canton_nom', 'epci_code', 'epci_nom', 'academie_code',
       'academie_nom', 'code_postal', 'codes_postaux', 'zone_emploi',
       'code_insee_centre_zone_emploi', 'code_unite_urbaine',
       'nom_unite_urbaine', 'taille_unite_urbaine',
       'type_commune_unite_urbaine', 'statut_commune_unite_urbaine',
       'population', 'superficie_hectare', 'superficie_km2', 'densite',
       'altitude_moyenne', 'altitude_minimale', 'altitude_maximale',
       'latitude_mairie', 'longitude_mairie', 'latitude_centre',
       'longitude_centre', 'grille_densite', 'grille_densite_texte',
       'niveau_equipements_services', 'niveau_equipements_services_texte',
       'gentile', 'url_wikipedia', 'url_villedereve'],
      dtype='str')

In [80]:
# Dictionnaire de correspondance (Ancien nom -> Nouveau nom pour tes analyses)
colonnes_mapping = {
    'code_insee': 'code_insee',
    'code_postal': 'localisation',       # Correspond au code postal principal
    'nom_standard': 'nom_standard',
    'reg_code': 'region',                # Renommé pour l'analyse
    'dep_code': 'departement',           # Renommé pour l'analyse
    'population': 'population',
    'superficie_hectare': 'superficie_hectare',
    'densite': 'densite',
    'altitude_moyenne': 'altitude_moyenne',
    'altitude_minimale': 'altitude_minimale',
    'altitude_maximale': 'altitude_maximale',
    'latitude_mairie': 'latitude_mairie',
    'longitude_mairie': 'longitude_mairie'
}

# filtre le DataFrame d'origine pour ne garder que ces colonnes et les renommer
df_filtre = df[list(colonnes_mapping.keys())].rename(columns=colonnes_mapping)

# Définit les types cibles compatibles avec PostgreSQL
# Utiliser des majuscules (Int32, Float64, string) permet à Pandas de gérer les valeurs absentes (NaN) sans bloquer
types_cibles = {
    'code_insee': 'string',
    'localisation': 'Int32',
    'nom_standard': 'string',
    'region': 'Int8',
    'departement': 'string',  # En string pour garder les préfixes ou formats spéciaux si besoin
    'population': 'Int32',
    'superficie_hectare': 'Int32',
    'densite': 'Float64',
    'altitude_moyenne': 'Int8',
    'altitude_minimale': 'Int8',
    'altitude_maximale': 'Int8',
    'latitude_mairie': 'Float64',
    'longitude_mairie': 'Float64'
}

# applique le typage au DataFrame
df = df_filtre.astype(types_cibles)

In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34935 entries, 0 to 34934
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   code_insee          34935 non-null  string 
 1   localisation        34932 non-null  Int32  
 2   nom_standard        34935 non-null  string 
 3   region              34935 non-null  Int8   
 4   departement         34935 non-null  string 
 5   population          34935 non-null  Int32  
 6   superficie_hectare  34935 non-null  Int32  
 7   densite             34932 non-null  Float64
 8   altitude_moyenne    34935 non-null  Int8   
 9   altitude_minimale   34935 non-null  Int8   
 10  altitude_maximale   34935 non-null  Int8   
 11  latitude_mairie     34935 non-null  Float64
 12  longitude_mairie    34935 non-null  Float64
dtypes: Float64(3), Int32(3), Int8(4), string(3)
memory usage: 3.1 MB


In [82]:
df.head()

,code_insee,localisation,nom_standard,region,departement,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie
0,01001,1400,L'Abergement-Clémenciat,84,01,832,1565,53.0,-14,-50,16,46.151,4.921
1,01002,1640,L'Abergement-de-Varey,84,01,267,912,29.0,-29,34,-20,46.007,5.423
2,01004,1500,Ambérieu-en-Bugey,84,01,14854,2448,607.0,123,-19,-15,45.958,5.36
3,01005,1330,Ambérieux-en-Dombes,84,01,1897,1605,118.0,34,9,46,45.996,4.903
4,01006,1300,Ambléon,84,01,113,602,19.0,77,74,-84,45.748,5.601


### Gestion des altitudes négatives

In [83]:
df_altitude_negative = df[df["altitude_maximale"] < 0]
df_altitude_negative

,code_insee,localisation,nom_standard,region,departement,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie
1,01002,1640,L'Abergement-de-Varey,84,01,267,912,29.0,-29,34,-20,46.007,5.423
2,01004,1500,Ambérieu-en-Bugey,84,01,14854,2448,607.0,123,-19,-15,45.958,5.36
4,01006,1300,Ambléon,84,01,113,602,19.0,77,74,-84,45.748,5.601
5,01007,1500,Ambronay,84,01,2833,3359,84.0,53,-31,-3,46.008,5.361
8,01010,1350,Anglefort,84,01,1125,2948,38.0,-62,-18,-12,45.914,5.809
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34925,97608,97615,Dzaoudzi,6,976,17831,650,2743.0,32,0,-124,-12.788,45.272
34929,97612,97630,Mtsamboro,6,976,7705,1588,485.0,-110,0,-32,-12.697,45.069
34932,97615,97615,Pamandzi,6,976,11442,426,2686.0,52,0,-54,-12.798,45.275
34933,97616,97640,Sada,6,976,11156,1085,1028.0,-126,0,-2,-12.847,45.106


### Détection des altitudes négatives illogiques

In [84]:
number_negative_max_alt = len(df[df["altitude_maximale"] < 0])
print(f'Il y a {number_negative_max_alt} altitude_max négative')

Il y a 16865 altitude_max négative


In [85]:
# condition illogiques
negative_max_alt = df["altitude_maximale"] < 0
wrong_min__max = df["altitude_minimale"] > df["altitude_maximale"]

# combinaison des deux conditions avec l'opérateur &
non_logical_filter = negative_max_alt & wrong_min__max

# Application du filtre sur le DataFrame
df_illogical_negative_alt = df[non_logical_filter]

print(f"Nombre de lignes incohérentes : {len(df_illogical_negative_alt)}")

Nombre de lignes incohérentes : 12729


In [86]:
# Afficher les colonnes clés pour inspecter visuellement
cols = ["nom_standard", "altitude_minimale", "altitude_maximale"]

# filtre l'affichage sur les colonnes existantes
colonnes_presentes = [col for col in cols if col in df.columns]

print(df_illogical_negative_alt[colonnes_presentes].head())

             nom_standard  altitude_minimale  altitude_maximale
1   L'Abergement-de-Varey                 34                -20
4                 Ambléon                 74                -84
11                Arandas                -57                -85
27              Bellignat                  4               -111
30                 Belley                -36               -120


### Visualisation cartographique pour déterminer les incohérences

In [91]:
# Conversion en GeoDataFrame
from shapely.geometry import Point
geometry = [Point(xy) for xy in zip(df.longitude_mairie, df.latitude_mairie)]
gdf_fire = gpd.GeoDataFrame(df, geometry=geometry)
# coordinate representation sytem
gdf_fire.crs = "EPSG:4326"

print(f"Dataset créé avec {len(gdf_fire)} points")
print("\nAperçu des données:")
gdf_fire.head()

Dataset créé avec 34935 points

Aperçu des données:


,code_insee,localisation,nom_standard,region,departement,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,geometry
0,01001,1400,L'Abergement-Clémenciat,84,01,832,1565,53.0,-14,-50,16,46.151,4.921,POINT (4.921 46.151)
1,01002,1640,L'Abergement-de-Varey,84,01,267,912,29.0,-29,34,-20,46.007,5.423,POINT (5.423 46.007)
2,01004,1500,Ambérieu-en-Bugey,84,01,14854,2448,607.0,123,-19,-15,45.958,5.36,POINT (5.36 45.958)
3,01005,1330,Ambérieux-en-Dombes,84,01,1897,1605,118.0,34,9,46,45.996,4.903,POINT (4.903 45.996)
4,01006,1300,Ambléon,84,01,113,602,19.0,77,74,-84,45.748,5.601,POINT (5.601 45.748)


In [92]:
## Création de la Heatmap des risques d'incendie

# Calcul du centre de la carte
center_lat = df['latitude_mairie'].mean()
center_lon = df['longitude_mairie'].mean()

# Création de la carte de base
m_heatmap = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=7,
    tiles='OpenStreetMap'
)

# Préparation des données pour la heatmap
# Chaque point contient [latitude, longitude, poids/intensité]
alt_data = [[row['latitude_mairie'], row['longitude_mairie'], row['altitude_maximale']]
             for idx, row in gdf_fire.iterrows()]

# Ajout de la heatmap à la carte
HeatMap(
    alt_data,
    min_opacity=0.2,
    radius=15,
    blur=15,
    max_zoom=1,
    gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'}
).add_to(m_heatmap)

# Ajout d'un titre à la carte
folium.map.Marker(
    [center_lat + 0.5, center_lon],
    icon=folium.DivIcon(
        html=f'<div style="font-size: 14px; font-weight: bold;">Heatmap des altitude maximalesdiv>',
        icon_size=(200, 20),
        icon_anchor=(0, 0)
    )
).add_to(m_heatmap)

print(f"Heatmap créée avec {len(alt_data)} points de données")
print(f"Centre de la carte: ({center_lat:.2f}, {center_lon:.2f})")

# Affichage de la carte
m_heatmap

Heatmap créée avec 34935 points de données
Centre de la carte: (46.79, 2.71)


In [ ]:

# ---------------------------------------------------------
# Étape 1 : Nettoyage et préparation des données
# ---------------------------------------------------------
colonnes_requises = ["latitude_mairie", "longitude_mairie", "altitude_maximale"]

# 1.1 Filtrer les lignes sans coordonnées ou sans altitude
df_clean = df.dropna(subset=colonnes_requises).copy()

# 1.2 Forcer la conversion en valeurs numériques
for col in colonnes_requises:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean = df_clean.dropna(subset=colonnes_requises)

# ---------------------------------------------------------
# Étape 2 : Normalisation du poids d'altitude (Min-Max Scaling)
# ---------------------------------------------------------
# Les poids d'une HeatMap doivent être strictement positifs
# On décale pour que la valeur minimale devienne 0
alt_min = df_clean["altitude_maximale"].min()
alt_max = df_clean["altitude_maximale"].max()


# Construction de la liste [latitude, longitude, poids_normalisé]
alt_data = df_clean[["latitude_mairie", "longitude_mairie", "alt_poids"]].values.tolist()

# ---------------------------------------------------------
# Étape 3 : Création de la carte Folium
# ---------------------------------------------------------
center_lat = df_clean["latitude_mairie"].mean()
center_lon = df_clean["longitude_mairie"].mean()

m_heatmap = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=6,
    tiles="OpenStreetMap"
)

# ---------------------------------------------------------
# Étape 4 : Configuration adaptée de la HeatMap
# ---------------------------------------------------------
HeatMap(
    alt_data,
    min_opacity=0.3,
    radius=12,          # Rayon d'influence de chaque commune
    blur=10,            # Flou pour adoucir les transitions
    max_val=1.0,        # Valeur maximale attendue (alignée avec notre normalisation)
    gradient={
        0.0: "#0000ff", # Bleu : altitudes les plus faibles / anomalies négatives
        0.3: "#00ffff", # Cyan
        0.6: "#00ff00", # Vert : moyennes altitudes
        0.8: "#ffff00", # Jaune
        1.0: "#ff0000"  # Rouge : hauts sommets
    }
).add_to(m_heatmap)

print(f"Points tracés : {len(alt_data)}")
print(f"Plage d'altitudes traitée : de {alt_min} m à {alt_max} m")

/tmp/ipykernel_27541/1377229011.py:24: RuntimeWarning: overflow encountered in scalar subtract
  df_clean["alt_poids"] = (df_clean["altitude_maximale"] - alt_min) / (alt_max - alt_min)


Points tracés : 34935
Plage d'altitudes traitée : de -128 m à 127 m


/tmp/ipykernel_27541/1377229011.py:44: UserWarning: The `max_val` parameter is no longer necessary. The largest intensity is calculated automatically.
  HeatMap(


In [ ]:
# Étape 1 : Isoler les cas impossibles selon la réalité physique


# En France, le point le plus bas naturel/aménagé est à environ -4 m (Les Moëres)
SEUIL_MIN_PHYSIQUE = -4

# Communes avec une altitude minimale ou maximale en dessous de ce seuil physique
df_anomalies_physiques = df[
    (df["altitude_minimale"] < SEUIL_MIN_PHYSIQUE) |
    (df["altitude_maximale"] < SEUIL_MIN_PHYSIQUE)
]

print(f"Nombre de communes sous -4 m : {len(df_anomalies_physiques)}")


# Étape 2 : Analyser la distribution des valeurs suspectes


print("\n--- Statistiques des altitudes maximales négatives ---")
print(df[df["altitude_maximale"] < 0]["altitude_maximale"].describe())

print("\n--- Échantillon de communes avec altitudes négatives ---")
cols_diagnostic = ["nom_standard", "altitude_minimale", "altitude_maximale"]
cols_dispos = [c for c in cols_diagnostic if c in df.columns]
print(df_anomalies_physiques[cols_dispos].head(10))


# Étape 3 : Tester l'hypothèse « simple inversion de signe »
# Si on prend la valeur absolue, est-ce que min reste <= max ?


df_test_abs = df_anomalies_physiques.copy()
df_test_abs["alt_min_abs"] = df_test_abs["altitude_minimale"].abs()
df_test_abs["alt_max_abs"] = df_test_abs["altitude_maximale"].abs()

# Cas où même en valeur absolue, le min dépasse le max :
incoherence_meme_apres_abs = df_test_abs[df_test_abs["alt_min_abs"] > df_test_abs["alt_max_abs"]]
print(f"\nCas où abs(min) > abs(max) (inversion de signe insuffisante) : {len(incoherence_meme_apres_abs)}")

Nombre de communes sous -4 m : 22829

--- Statistiques des altitudes maximales négatives ---
count      16865.0
mean    -73.357189
std      36.612757
min         -128.0
25%         -105.0
50%          -78.0
75%          -43.0
max           -1.0
Name: altitude_maximale, dtype: Float64

--- Échantillon de communes avec altitudes négatives ---
               nom_standard  altitude_minimale  altitude_maximale
0   L'Abergement-Clémenciat                -50                 16
1     L'Abergement-de-Varey                 34                -20
2         Ambérieu-en-Bugey                -19                -15
4                   Ambléon                 74                -84
5                  Ambronay                -31                 -3
6                  Ambutrix                -19                114
7          Andert-et-Condon                -31                118
8                 Anglefort                -18                -12
9                  Apremont               -110                 

In [ ]:
# Exemple de correction ciblée avec .loc :
# df.loc[filtre_illogique, "altitude_maximale"] = df.loc[filtre_illogique, "altitude_maximale"].abs()

## Export

In [ ]:
df.to_parquet(
    GEO_DATA_CLEAN_DIR / "communes.parquet",
    index=False
)